# 05 · Market Alignment
**MLS NLP Pipeline — Stage 5 (Business Extension)**

Compares **narrative capital** (media prominence) against **commercial market value**
(Forbes franchise valuations) to surface business opportunities.

### Framework

```
                    High Narrative
                          │
    OVEREXPOSED           │         MARKET LEADERS
    High narrative,       │         High narrative +
    lower valuation       │         high valuation
    ──────────────────────┼──────────────────────
    SLEEPING ASSETS       │         UNDEREXPOSED
    Low narrative +       │         High valuation,
    low valuation         │         lower narrative
                          │
                    Low Narrative
```

### Business Questions Answered
1. Which clubs are underpriced sponsorship assets?
2. Where does franchise value outpace brand/media presence?
3. Did the Messi signing at Inter Miami create measurable market value?
4. Which clubs most efficiently convert narrative into commercial value?

> **Data Note:** Forbes valuations are approximate figures based on publicly
> reported annual MLS franchise valuations. Verify before formal use.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
from scipy import stats

from pipeline.utils import get_config, PROJECT_ROOT

settings = get_config("settings")
data_dir = PROJECT_ROOT / settings["pipeline"]["data_dir"]

## 2. Load Combined Dataset

In [ ]:
df = pd.read_csv(data_dir / "analysis" / "press" / "market_alignment.csv")
print(f"{len(df)} club-year records | {df['entity'].nunique()} clubs | {df['year'].nunique()} years")
df.head(8)

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMarket gap = valuation_rank − narrative_rank")
print("  Positive → overexposed (narrative rank better than valuation rank)")
print("  Negative → underexposed (valuation rank better than narrative rank)")
print()
print(df[["entity", "year", "narrative_rank", "valuation_rank", "market_gap",
          "valuation_usd_m"]].sample(8).to_string(index=False))

## 3. Overall Market Gap — All Clubs

In [ ]:
CLUB_COLORS = {
    "Atlanta United FC": "#80000A", "Austin FC": "#00B140",
    "CF Montreal": "#003DA5",       "Charlotte FC": "#1A85C8",
    "Chicago Fire FC": "#CC0000",   "Colorado Rapids": "#862633",
    "Columbus Crew": "#FFF200",     "D.C. United": "#000000",
    "FC Dallas": "#BF0D3E",         "Houston Dynamo FC": "#F68712",
    "Inter Miami CF": "#F7B5CD",    "LA Galaxy": "#00245D",
    "LAFC": "#C39E6D",              "Minnesota United FC": "#8CD2F4",
    "Nashville SC": "#ECE83A",      "New England Revolution": "#0A2240",
    "New York City FC": "#6CACE4",  "New York Red Bulls": "#ED1E36",
    "Orlando City SC": "#612B9B",   "Philadelphia Union": "#071B2C",
    "Portland Timbers": "#00482B",  "Real Salt Lake": "#B30838",
    "San Jose Earthquakes": "#0D4C92", "Seattle Sounders FC": "#5D9741",
    "Sporting Kansas City": "#93B8E3", "St. Louis City SC": "#DD0000",
    "Toronto FC": "#B81137",        "Vancouver Whitecaps FC": "#009BC8",
}

avg = df.groupby("entity")["market_gap"].mean().sort_values()

fig, ax = plt.subplots(figsize=(13, 9))
colors = [CLUB_COLORS.get(c, "#888") for c in avg.index]
bars = ax.barh(avg.index, avg.values, color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", lw=1.2)

for bar, val in zip(bars, avg.values):
    sign = "+" if val >= 0 else ""
    ax.text(val + (0.15 if val >= 0 else -0.15), bar.get_y() + bar.get_height() / 2,
            f"{sign}{val:.1f}", va="center", ha="left" if val >= 0 else "right", fontsize=8)

green_patch = mpatches.Patch(color="#27ae60", label="Overexposed — narrative > value (underpriced media asset)")
red_patch   = mpatches.Patch(color="#e74c3c", label="Underexposed — value > narrative (marketing gap)")
ax.legend(handles=[green_patch, red_patch], loc="lower right", fontsize=9)
ax.set_xlabel("Avg Market Gap (valuation rank − narrative rank)", fontsize=11)
ax.set_title("Market Alignment Gap — 2018–2024 Average", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Quadrant Analysis — Narrative vs. Valuation Rank

In [ ]:
avg2 = df.groupby("entity").agg(
    narrative_rank=("narrative_rank", "mean"),
    valuation_rank=("valuation_rank", "mean"),
    valuation_usd_m=("valuation_usd_m", "mean"),
).reset_index()

n_clubs = avg2["narrative_rank"].max()
mid     = n_clubs / 2

fig, ax = plt.subplots(figsize=(12, 10))
ax.set_facecolor("#f8f8f8")

# Quadrant shading
ax.axhspan(0, mid, xmin=0, xmax=0.5, alpha=0.06, color="green")
ax.axhspan(mid, n_clubs + 1, xmin=0.5, xmax=1.0, alpha=0.06, color="orange")
ax.axhline(mid, color="gray", lw=0.8, ls="--", alpha=0.5)
ax.axvline(mid, color="gray", lw=0.8, ls="--", alpha=0.5)

for _, row in avg2.iterrows():
    color = CLUB_COLORS.get(row["entity"], "#888")
    size  = np.clip(row["valuation_usd_m"] / 8, 30, 220)
    ax.scatter(row["narrative_rank"], row["valuation_rank"],
               s=size, color=color, edgecolors="white", linewidths=0.8, zorder=3, alpha=0.9)
    label = row["entity"].replace(" FC", "").replace(" CF", "").replace(" SC", "")
    ax.annotate(label, (row["narrative_rank"], row["valuation_rank"]),
                fontsize=7.5, ha="center", va="bottom",
                xytext=(0, 6), textcoords="offset points")

ax.plot([1, n_clubs], [1, n_clubs], color="gray", lw=1, ls=":", alpha=0.6)
ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel("Narrative Rank (1 = most media attention)", fontsize=11)
ax.set_ylabel("Valuation Rank (1 = highest Forbes value)", fontsize=11)
ax.set_title("Narrative Capital vs. Commercial Value — 2018–2024 Avg", fontsize=13, fontweight="bold")

ax.text(n_clubs*0.85, n_clubs*0.1, "UNDEREXPOSED
High value, low narrative
→ Marketing opportunity",
        fontsize=8, color="darkorange", alpha=0.8, ha="center")
ax.text(n_clubs*0.15, n_clubs*0.9, "OVEREXPOSED
High narrative, lower value
→ Sponsorship opportunity",
        fontsize=8, color="green", alpha=0.8, ha="center")
ax.text(n_clubs*0.15, n_clubs*0.1, "MARKET LEADERS
High narrative + high value",
        fontsize=8, color="#333", alpha=0.6, ha="center")
plt.tight_layout()
plt.show()

## 5. Case Study — Inter Miami & the Messi Effect

In [ ]:
miami = df[df["entity"] == "Inter Miami CF"].sort_values("year")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

color = "#F7B5CD"
ax1.bar(miami["year"], miami["valuation_usd_m"], alpha=0.5, color=color, label="Valuation ($M)")
ax2.plot(miami["year"], miami["narrative_rank"], marker="o", lw=2.5, color="#c0392b", label="Narrative Rank")
ax2.invert_yaxis()

ax1.set_ylabel("Forbes Valuation ($M)", fontsize=11)
ax2.set_ylabel("Narrative Rank (lower = more prominent)", fontsize=11)
ax1.set_xlabel("Year")
ax1.set_title("Inter Miami CF — Valuation vs. Narrative Rank", fontsize=13, fontweight="bold")

# Messi annotation
ax1.annotate("Messi signs\n(Jul 2023)", xy=(2023, miami[miami["year"]==2023]["valuation_usd_m"].values[0]),
             xytext=(2021.5, 800), fontsize=9, color="black",
             arrowprops=dict(arrowstyle="->", color="black"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

print("\nInter Miami — year-by-year:")
print(miami[["year", "valuation_usd_m", "narrative_rank", "market_gap"]].to_string(index=False))
print(f"\nValuation jump 2022→2023: ${miami[miami['year']==2023]['valuation_usd_m'].values[0] - miami[miami['year']==2022]['valuation_usd_m'].values[0]:.0f}M (+{(miami[miami['year']==2023]['valuation_usd_m'].values[0] / miami[miami['year']==2022]['valuation_usd_m'].values[0] - 1)*100:.0f}%)")

## 6. Narrative-to-Value Conversion Efficiency

In [ ]:
avg3 = df.groupby("entity").agg(
    avg_narrative_rank=("narrative_rank", "mean"),
    avg_valuation=("valuation_usd_m", "mean"),
).reset_index()

n = avg3["avg_narrative_rank"].max()
avg3["narrative_prominence"] = n + 1 - avg3["avg_narrative_rank"]
avg3["value_per_prominence"]  = avg3["avg_valuation"] / avg3["narrative_prominence"]
avg3 = avg3.sort_values("value_per_prominence", ascending=False)

fig, ax = plt.subplots(figsize=(13, 9))
colors = [CLUB_COLORS.get(c, "#888") for c in avg3["entity"]]
bars = ax.barh(avg3["entity"], avg3["value_per_prominence"], color=colors, edgecolor="white")

for bar, val in zip(bars, avg3["value_per_prominence"]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f"${val:.0f}M", va="center", fontsize=8)

ax.set_xlabel("Avg Valuation ($M) per Narrative Prominence Unit", fontsize=11)
ax.set_title("Which Clubs Extract the Most Commercial Value from Their Media Presence?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Correlation: Does Narrative Predict Valuation?

In [ ]:
r, p = stats.pearsonr(df["narrative_rank"], df["valuation_rank"])
print(f"Pearson r (narrative rank vs valuation rank): {r:.3f}  (p={p:.4f})")
print()

r2, p2 = stats.pearsonr(df["pagerank"], df["valuation_usd_m"])
print(f"Pearson r (PageRank vs valuation $M): {r2:.3f}  (p={p2:.4f})")
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df["narrative_rank"], df["valuation_rank"], alpha=0.3, color="#3498db", s=25)
m, b = np.polyfit(df["narrative_rank"], df["valuation_rank"], 1)
x_line = np.linspace(df["narrative_rank"].min(), df["narrative_rank"].max(), 100)
axes[0].plot(x_line, m * x_line + b, color="red", lw=2)
axes[0].set_xlabel("Narrative Rank"); axes[0].set_ylabel("Valuation Rank")
axes[0].set_title(f"Rank Correlation  (r={r:.2f})")
axes[0].invert_xaxis(); axes[0].invert_yaxis()

axes[1].scatter(df["pagerank"], df["valuation_usd_m"], alpha=0.3, color="#e74c3c", s=25)
m2, b2 = np.polyfit(df["pagerank"], df["valuation_usd_m"], 1)
x2 = np.linspace(df["pagerank"].min(), df["pagerank"].max(), 100)
axes[1].plot(x2, m2 * x2 + b2, color="blue", lw=2)
axes[1].set_xlabel("PageRank"); axes[1].set_ylabel("Valuation ($M)")
axes[1].set_title(f"PageRank vs. Valuation  (r={r2:.2f})")

plt.suptitle("Does Narrative Capital Predict Commercial Value?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Business Takeaways

| Stakeholder | Signal to Watch | Action |
|-------------|----------------|--------|
| **Sponsor / Brand** | Club where narrative rank > valuation rank | Buy jersey/naming rights before market reprices |
| **Club Front Office** | Your valuation rank > narrative rank | Invest in content/PR to close the gap |
| **Investor / PE Firm** | Rising narrative momentum + lagging valuation | Acquire before the market catches up |
| **Broadcaster** | Narrative centrality spikes by region | Weight rights bids by centrality, not just market size |
| **Player Agent** | Club in narrative spike quarter | Negotiate contracts during peak attention windows |

### Key Numbers from This Dataset
- **CF Montreal**: narrative rank #3.9 avg, valuation rank #22 → most underpriced media asset
- **Atlanta United**: valuation rank #3.4 avg, narrative rank #14.3 → biggest marketing gap
- **Inter Miami 2023**: +$685M valuation jump, directly tied to reaching narrative rank #1
- **Seattle Sounders**: most efficient converter — high value AND high narrative, tightly aligned